# Hand Gesture Recognition for Robot Control

Used to recognize hand gestures from a webcam and uses them as simple
robot commands. The hand is found with Google's MediaPipe library, which gives
21 landmark points (x, y, z). Then those points are used to train a classical classifier (Random Forest and SVM) to tell
the gestures apart.

The gestures are: open_palm, fist, thumbs_up, point and peace.

**Completed**: set up MediaPipe and show the detected hand landmarks.

## 1. Detecting the hand with MediaPipe

MediaPipe already knows how to find a hand, so we use it as-is to get the 21
landmarks. We only need a few small helper functions: download the model,
create the detector, turn a detection into a NumPy array and draw it.

In [1]:
import os
import time
import urllib.request

import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt

In [2]:
# the pre-trained model and the 21 bones we draw between the landmarks
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/1/hand_landmarker.task")
MODEL_PATH = "hand_landmarker.task"

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # index
    (5, 9), (9, 10), (10, 11), (11, 12),    # middle
    (9, 13), (13, 14), (14, 15), (15, 16),  # ring
    (13, 17), (17, 18), (18, 19), (19, 20), # pinky
    (0, 17),                                # palm base
]


def download_model():
    """Download the MediaPipe hand model the first time we need it."""
    if not os.path.exists(MODEL_PATH):
        print("Downloading the hand model...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    return MODEL_PATH


def create_landmarker(mode="video"):
    """Create a MediaPipe hand landmarker.

    mode: "video" for the webcam, "image" for a single picture.
    Returns the landmarker object.
    """
    download_model()
    running = vision.RunningMode.VIDEO if mode == "video" else vision.RunningMode.IMAGE
    options = vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=running, num_hands=1)
    return vision.HandLandmarker.create_from_options(options)


def landmarks_to_array(hand):
    """Turn one detected hand (21 landmarks) into a (21, 3) NumPy array."""
    return np.array([[p.x, p.y, p.z] for p in hand], dtype=np.float32)


def draw_landmarks(image, hands):
    """Draw the landmark points and bones on a BGR image (in place)."""
    h, w = image.shape[:2]
    for hand in hands:
        pts = [(int(p.x * w), int(p.y * h)) for p in hand]
        for a, b in HAND_CONNECTIONS:
            cv2.line(image, pts[a], pts[b], (255, 255, 255), 2)
        for (x, y) in pts:
            cv2.circle(image, (x, y), 4, (0, 0, 255), -1)
    return image

This function opens the camera and draws the landmarks live; press **q** to close it.

In [3]:
def run_hand_tracking():
    """Open the webcam and draw the hand landmarks live (press q to quit)."""
    detector = create_landmarker("video")
    cap = cv2.VideoCapture(0)
    start, last = time.perf_counter(), -1
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ts = max(last + 1, int((time.perf_counter() - start) * 1000)); last = ts
        result = detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)
        draw_landmarks(frame, result.hand_landmarks)
        cv2.imshow("hand tracking (press q)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release(); cv2.destroyAllWindows(); detector.close()

In [4]:
run_hand_tracking()